# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nfatima25seecs/ml-pipeline-ex/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule:** A page is flagged if `trend_direction == 'down'` and
`impressions_last_30d > 0`. The action score scales with impression
tier (how visible the page is) and `trend_pct` (how sharp the drop is).
High-visibility pages with the sharpest drops get the highest priority.

**Three reason codes:**
- `declining_high_visibility` → excellent impression tier, score 0.80-1.00 → REFRESH_CONTENT
- `declining_moderate_visibility` → good impression tier, score 0.50-0.79 → REVIEW_SERP  
- `declining_low_visibility` → low/moderate impression tier, score 0.20-0.49 → MONITOR

**Why these signals?**
- `trend_direction` confirms the page is actively losing traffic right now.
- `impression_tier` tells us how much is at stake — a declining page with
  7,000 impressions costs far more to ignore than one with 10.
- `trend_pct` measures severity — a -95% drop is more urgent than -21%.

In [22]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("/content_refresh_anonymized.csv")

# ── SIGNAL CHECK 1: trend_direction ──────────────────────────────────
print("=== Signal 1: trend_direction ===")
print(df['trend_direction'].value_counts())
down_pct = round((df['trend_direction'] == 'down').sum() / len(df) * 100, 1)
print(f"VERDICT: CONFIRMED — {down_pct}% of pages are declining\n")

# ── SIGNAL CHECK 2: impression tier for declining pages ──────────────
print("=== Signal 2: impression tier for declining pages ===")
bucket = (df[df['trend_direction'] == 'down']
          .groupby('impression_tier')
          .size()
          .reset_index(name='n'))
print(bucket)
print("VERDICT: CONFIRMED — excellent/good tier pages are the highest-value targets")

=== Signal 1: trend_direction ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
VERDICT: CONFIRMED — 54.2% of pages are declining

=== Signal 2: impression tier for declining pages ===
  impression_tier     n
0       excellent   498
1            good  4223
2             low  5106
3        moderate  6435
VERDICT: CONFIRMED — excellent/good tier pages are the highest-value targets


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
# ── BUILD THE SCORE ───────────────────────────────────────────────────
scored = df.copy()
scores = np.zeros(len(scored))
reason_codes = np.full(len(scored), "stable_or_growing", dtype=object)
action_labels = np.full(len(scored), "NO_ACTION", dtype=object)

# Base condition: must be declining with real traffic
is_down = (scored['trend_direction'] == 'down') & (scored['impressions_last_30d'] > 0)

# Tier 1 — excellent impression + declining → REFRESH_CONTENT
cond_high = is_down & (scored['impression_tier'] == 'excellent')
severity = np.abs(scored.loc[cond_high, 'trend_pct']) / 100
scores[cond_high] = np.clip(0.80 + (severity * 0.20), 0.80, 1.00)
reason_codes[cond_high] = "declining_high_visibility"
action_labels[cond_high] = "REFRESH_CONTENT"

# Tier 2 — good impression + declining → REVIEW_SERP
cond_mod = is_down & (scored['impression_tier'] == 'good')
severity2 = np.abs(scored.loc[cond_mod, 'trend_pct']) / 100
scores[cond_mod] = np.clip(0.50 + (severity2 * 0.29), 0.50, 0.79)
reason_codes[cond_mod] = "declining_moderate_visibility"
action_labels[cond_mod] = "REVIEW_SERP"

# Tier 3 — low/moderate impression + declining → MONITOR
cond_low = is_down & (scored['impression_tier'].isin(['low', 'moderate']))
severity3 = np.abs(scored.loc[cond_low, 'trend_pct']) / 100
scores[cond_low] = np.clip(0.20 + (severity3 * 0.29), 0.20, 0.49)
reason_codes[cond_low] = "declining_low_visibility"
action_labels[cond_low] = "MONITOR"

scored['action_score'] = np.round(scores, 3)
scored['reason_code'] = reason_codes
scored['action_label'] = action_labels

# ── RANKED QUEUE ──────────────────────────────────────────────────────
queue = (scored[scored['action_label'] != 'NO_ACTION']
         .sort_values('action_score', ascending=False))

queue = queue[['content_id', 'impression_tier', 'impressions_last_30d',
               'trend_pct', 'action_score', 'reason_code', 'action_label']]

print(f"Total flagged pages: {len(queue):,}")
print("\nTop 20:")
print(queue.head(20).to_string())

# ── WRITE CSV ─────────────────────────────────────────────────────────
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nCSV written.")

Total flagged pages: 14,867

Top 20:
                 content_id impression_tier  impressions_last_30d  trend_pct  action_score                reason_code     action_label
14955  content_ec66c58d9826       excellent                  2779      -95.2         0.990  declining_high_visibility  REFRESH_CONTENT
7122   content_7a6df559322d       excellent                   462      -94.7         0.989  declining_high_visibility  REFRESH_CONTENT
11577  content_20e876d26019       excellent                  1716      -94.0         0.988  declining_high_visibility  REFRESH_CONTENT
17412  content_3437133c7ccf       excellent                  2699      -92.6         0.985  declining_high_visibility  REFRESH_CONTENT
8876   content_4092ad6d1f71       excellent                   840      -90.2         0.980  declining_high_visibility  REFRESH_CONTENT
9412   content_8053a66bd6ac       excellent                  2472      -89.8         0.980  declining_high_visibility  REFRESH_CONTENT
15968  content_66b

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 Review Summary**

Assigned Action & Reason Code: All top 20 flagged pages are assigned action_label = REFRESH_CONTENT with reason_code = declining_high_visibility (action scores ranging from 0.964 to 0.990).

**Confidence Note:** High confidence. Every page in this tier belongs to the excellent impression tier (ranging up to 8,099 impressions) and exhibits severe traffic loss (declines between -81.8% and -95.2%). Prioritizing content refreshes on these high-visibility pages yields the highest return on SEO effort.


**Scenarios where recommendations could be wrong:**

**Seasonal Demand:** Pages covering highly seasonal topics (e.g., holiday promotions or annual events) will show sharp 30-day traffic drops that reflect natural off-peak interest rather than content decay.

 **Intentional Migration or Consolidation:**
 Pages scheduled for deprecation, canonical redirection, or merger into new parent URLs will naturally show declining impression metrics.

**Algorithm / SERP Layout Shifts:**
Structural SERP changes (e.g., AI Overviews or featured snippets eating click-through rates) may cause impression drops even if the underlying content remains accurate and high quality.

In [24]:
# Display top 20 summary statistics for written review
top_20 = queue.head(20)

print("=== TOP 20 ACTION & REASON SUMMARY ===")
print(top_20[['action_label', 'reason_code']].value_counts())

print("\n=== TOP 20 METRICS SUMMARY ===")
print(f"Min Score: {top_20['action_score'].min()} | Max Score: {top_20['action_score'].max()}")
print(f"Min Drop %: {top_20['trend_pct'].min()}% | Max Drop %: {top_20['trend_pct'].max()}%")
print(f"Min Impressions: {top_20['impressions_last_30d'].min():,} | Max Impressions: {top_20['impressions_last_30d'].max():,}")

=== TOP 20 ACTION & REASON SUMMARY ===
action_label     reason_code              
REFRESH_CONTENT  declining_high_visibility    20
Name: count, dtype: int64

=== TOP 20 METRICS SUMMARY ===
Min Score: 0.964 | Max Score: 0.99
Min Drop %: -95.2% | Max Drop %: -81.8%
Min Impressions: 462 | Max Impressions: 8,099


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis**

**Potential Anomalies:**
 content_7a6df559322d (Rank 2, score 0.989) and content_4092ad6d1f71 (Rank 5, score 0.980) have relatively low absolute impression volume (462 and 840, respectively) compared to heavyweights like content_73e5410650b9 (8,099 impressions). They were ranked near the top purely due to severe percentage drops (-94.7% and -90.2%) within the excellent tier boundary. Adding a secondary sort by absolute impression volume (impressions_last_30d) would refine priority order.


**Leakage & Bias Verification**

No Future Leakage: The scoring rule relies strictly on past 30-day window metrics (trend_direction, impression_tier, impressions_last_30d, and trend_pct). No future traffic metrics, conversion windows, or forward-looking product flags were included in the decision logic.

In [25]:
# 1. Identify low impression anomalies ranked in the top 20
print("=== WEAK PICKS / LOW IMPRESSION CHECK (Top 20) ===")
low_imp_top20 = top_20[top_20['impressions_last_30d'] < 1000]
print(low_imp_top20[['content_id', 'impressions_last_30d', 'trend_pct', 'action_score']])

# 2. Check for potential feature leakage or missing columns
print("\n=== LEAKAGE CHECK ===")
used_cols = ['trend_direction', 'impression_tier', 'impressions_last_30d', 'trend_pct']
leaked_cols = [c for c in df.columns if 'future' in c or 'convert' in c or 'next' in c]

print(f"Columns used in rule: {used_cols}")
print(f"Potential leakage columns found in dataset: {leaked_cols if leaked_cols else 'None'}")

=== WEAK PICKS / LOW IMPRESSION CHECK (Top 20) ===
                content_id  impressions_last_30d  trend_pct  action_score
7122  content_7a6df559322d                   462      -94.7         0.989
8876  content_4092ad6d1f71                   840      -90.2         0.980

=== LEAKAGE CHECK ===
Columns used in rule: ['trend_direction', 'impression_tier', 'impressions_last_30d', 'trend_pct']
Potential leakage columns found in dataset: None


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.